# Rule of Thumb on HateXPlain: explanations vs human rationales

This notebook fits a text RoT explainer (`fit_text` over ModernBERT
embeddings) to a hate-speech black box and gauges the explanations against
human rationale spans three ways: **plausibility** (length-weighted AUROC,
the legacy-endorsed metric), **faithfulness** (deletion/insertion vs random)
and **bias slices** (fidelity per target community).

Human rationales record what annotators found convincing (plausibility),
not what the model computed — so they sit alongside intervention-based
faithfulness, never as the only score. Everything runs on CPU; the only
downloads are two small JSON files plus cached model weights.

In [1]:
import json
import os
import urllib.request
from collections import Counter

import numpy as np

CACHE = "./_hx_cache"
os.makedirs(CACHE, exist_ok=True)
BASE = "https://raw.githubusercontent.com/punyajoy/HateXplain/master/Data/"
for name in ("dataset.json", "post_id_divisions.json"):
    dest = os.path.join(CACHE, name)
    if not os.path.exists(dest):
        print(f"downloading {name} ...", flush=True)
        urllib.request.urlretrieve(BASE + name, dest)
print("cached:", sorted(os.listdir(CACHE)))

cached: ['dataset.json', 'post_id_divisions.json']


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

LAB = {"hatespeech": 1, "offensive": 1, "normal": 0}
with open(os.path.join(CACHE, "dataset.json")) as f:
    ds = json.load(f)
with open(os.path.join(CACHE, "post_id_divisions.json")) as f:
    div = json.load(f)

def parse(pid):
    v = ds[pid]
    toks = v["post_tokens"]
    votes = [LAB[a["label"]] for a in v["annotators"]]
    c = Counter(votes)
    if c[0] == c[1]:
        return None
    fixed = []
    for r in v.get("rationales") or []:
        r = list(r)[:len(toks)] + [0] * max(0, len(toks) - len(r))
        fixed.append(r)
    rat = np.mean(fixed, axis=0) if fixed else np.zeros(len(toks))
    tgts = [t for a in v["annotators"] for t in a.get("target", [])]
    return " ".join(toks), 1 if c[1] > c[0] else 0, rat, Counter(tgts).most_common(1)[0][0] if tgts else "None"

splits = {}
for split in ("train", "test"):
    T, Y, R, G = [], [], [], []
    for pid in div[split]:
        r = parse(pid)
        if r is not None:
            t, y, rat, tgt = r
            T.append(t); Y.append(y); R.append(rat); G.append(tgt)
    splits[split] = (T, np.array(Y), R, G)
    print(f"{split}: n={len(T)}", flush=True)

vec = TfidfVectorizer(max_features=8000, token_pattern=r"(?u)\b\w+\b")
Xtr = vec.fit_transform(splits["train"][0])
bb = LogisticRegression(max_iter=500).fit(Xtr, splits["train"][1])
for split in ("train", "test"):
    T, Y, _, _ = splits[split]
    print(f"black box vs humans ({split}): {bb.score(vec.transform(T), Y):.3f}", flush=True)
ybb_tr, ybb_te = bb.predict(Xtr), bb.predict(vec.transform(splits["test"][0]))

train: n=15383


test: n=1924


black box vs humans (train): 0.832


black box vs humans (test): 0.759


/Users/bigcamel/Oxford/Projects/Rule-of-Thumb-Explaining-Artificial-Intelligence-Systems-using-Partial-Information/Rule-of-Thumb/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: divide by zero encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)
/Users/bigcamel/Oxford/Projects/Rule-of-Thumb-Explaining-Artificial-Intelligence-Systems-using-Partial-Information/Rule-of-Thumb/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: overflow encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)
/Users/bigcamel/Oxford/Projects/Rule-of-Thumb-Explaining-Artificial-Intelligence-Systems-using-Partial-Information/Rule-of-Thumb/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: invalid value encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weigh

## Fit RoT on a training slice

`fit_text` learns per-embedding-dim slopes shared across tokens — a linear
model on token-mean embeddings. Word order is invisible to it (see README
Limitations), so this is a lexical explainer: the honest question is
whether it recovers the lexical cues the TF-IDF box leans on. We fit a seed-fixed 3000-post slice, 100 epochs (the integration tests
use the canonical 300-epoch run of the same slice: fidelity 0.76,
weighted AUROC 0.71).

In [3]:
import ruleofthumb as rot

rng = np.random.RandomState(0)
take = rng.choice(len(splits["train"][0]), 3000, replace=False)
take.sort()
str_tr = [splits["train"][0][i] for i in take]
emb_tr = rot.embed_texts(str_tr, max_length=96, batch_size=32)
exp = rot.fit_text(ybb_tr[take], emb_tr.embeddings, mask=emb_tr.attention_mask,
                   epochs=100, batch_size=256, learning_rate=0.05, seed=0)
print(f"train agreement: {exp.train_agreement_:.3f}", flush=True)

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


train agreement: 0.801


## Plausibility: word-level AUROC vs human rationales

Subword scores are mean-pooled to whitespace words (special tokens never
overlap a word span), signed so positive always supports the predicted
class, and scored per post with AUROC — the metric the research code
endorses over top-k F1 (F1 is sensitive to annotation density). Random word
order, scored on the same posts, is the baseline.

In [4]:
from sklearn.metrics import roc_auc_score
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")
tte, rats, tgts = splits["test"][0], splits["test"][2], splits["test"][3]
emb_te = rot.embed_texts(tte, max_length=96, batch_size=32)
pred = np.asarray(exp.predict(emb_te.embeddings, mask=emb_te.attention_mask, sample_chunk=300))
print(f"test fidelity vs black box: {np.mean(pred == ybb_te):.3f}", flush=True)
imp = exp.get_explanation(emb_te.embeddings, mask=emb_te.attention_mask, sample_chunk=300)

def word_scores(row, text, offs):
    words = text.split()
    bounds, s = [], 0
    for w in words:
        s = text.index(w, s); bounds.append((s, s + len(w))); s += len(w)
    ws = np.zeros(len(words))
    for j, (a, b) in enumerate(bounds):
        vals = [row[k] for k, (x, y) in enumerate(offs) if y > a and x < b and not (x == 0 and y == 0)]
        ws[j] = float(np.mean(vals)) if vals else 0.0
    return ws

valid, aucs, w = [], [], []
pairs = []
for i, t in enumerate(tte):
    offs = tok(t, return_offsets_mapping=True, truncation=True, max_length=96)["offset_mapping"]
    ws = word_scores(imp[i], t, offs)
    pairs.append((ws, int(pred[i])))
    r = np.asarray(rats[i], dtype=float)
    if len(ws) == len(r):
        gt = (r > 0.5).astype(int)
        if gt.min() != gt.max():
            valid.append((ws if pred[i] == 1 else -ws, gt, len(r)))
aucs = np.array([roc_auc_score(g, s) for s, g, _ in valid])
w = np.array([n for _, _, n in valid], dtype=float)
rr = np.random.RandomState(0)
rand = np.array([roc_auc_score(g, rr.random(len(g))) for _, g, _ in valid])
print(f"posts scored: {len(valid)}", flush=True)
print(f"RoT weighted AUROC: {(aucs * w).sum() / w.sum():.3f} (mean {aucs.mean():.3f})", flush=True)
print(f"random baseline: {rand.mean():.3f}", flush=True)
assert (aucs * w).sum() / w.sum() > rand.mean() + 0.08

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


test fidelity vs black box: 0.758


posts scored: 1098


RoT weighted AUROC: 0.645 (mean 0.644)


random baseline: 0.487


## Faithfulness: deletion and insertion vs random

Delete (or insert) words most-important-first and re-query the black box: a
faithful ranking moves the predicted-class probability fast on deletion and
recovers it fast on insertion. Insertion is ordered by *signed* pro-class
weight (top-|imp| keeps negative evidence too — a harness lesson from the
research sandbox). Random word order is the baseline.

In [5]:
sel = np.random.RandomState(0).choice(len(tte), 200, replace=False)
Xte = vec.transform(tte)
p0 = bb.predict_proba(Xte[sel])
c = bb.predict(Xte[sel])
print("k  delete_RoT delete_rand insert_RoT insert_rand", flush=True)
for k in (3, 10, 30):
    dr, dn, ir, inn = [], [], [], []
    for ii, i in enumerate(sel):
        ws, p = pairs[i]
        words = tte[i].split()
        o = set(np.argsort(-np.abs(ws))[:k])
        sgn = ws if p == 1 else -ws
        so = set(np.argsort(-sgn)[:k])
        ro = set(np.random.RandomState(ii).permutation(len(words))[:k])
        dall = set(range(len(words)))
        for store, kept in ((dr, dall - o), (dn, dall - ro), (ir, so), (inn, ro)):
            txt = " ".join(w for j, w in enumerate(words) if j in kept)
            p1 = bb.predict_proba(vec.transform([txt]))[0, c[ii]]
            store.append(abs(p0[ii, c[ii]] - p1))
    print(f"{k}  {np.mean(dr):.4f} {np.mean(dn):.4f} {np.mean(ir):.4f} {np.mean(inn):.4f}", flush=True)

k  delete_RoT delete_rand insert_RoT insert_rand


3  0.1171 0.0649 0.1550 0.1830


10  0.1815 0.1482 0.0863 0.0921


30  0.2101 0.2027 0.0250 0.0180


## Gallery and coherence audit

Example highlights, the top positive terms per predicted class, the share
of importance mass sitting on stopwords, and fidelity sliced by target
community.

In [6]:
from ruleofthumb import plot

for cls, name in ((1, "abusive"), (0, "normal")):
    j = next(j for j, t in enumerate(tte) if pred[j] == cls)
    ws, _ = pairs[j]
    plot.text_html(ws, tte[j].split())
    print(f"[{name}] {' '.join(tte[j].split()[:24])}", flush=True)

[abusive] an these nigger biches look like godzilla nasty


[normal] <user> men can not be raped can not be abused that why they call it violence against women and children because men are always


In [7]:
STOP = {"i", "me", "my", "we", "our", "you", "he", "him", "his", "she", "her", "it", "they", "them", "this", "that", "these", "those", "is", "are", "was", "were", "be", "been", "am", "the", "a", "an", "and", "or", "of", "to", "in", "on", "for", "with", "as", "at", "by", "from"}
for cls in (1, 0):
    agg = Counter()
    for (ws, p), t in zip(pairs, tte):
        if p == cls:
            for w, v in zip(t.split(), ws):
                if v > 0:
                    agg[w.lower()] += v
    print(f"top class-{cls}: {[w for w, _ in agg.most_common(10)]}", flush=True)
num, den = 0.0, 0.0
for (ws, _), t in zip(pairs, tte):
    a = np.abs(ws)
    den += a.sum()
    num += sum(v for w, v in zip(t.split(), a) if w.lower() in STOP)
print(f"stopword mass share: {num / den:.3f}", flush=True)
print("fidelity by target:", flush=True)
for g, n in Counter(tgts).most_common():
    if n >= 30:
        m = np.array([t == g for t in tgts])
        print(f"  {g}: n={m.sum()} fidelity={np.mean(pred[m] == ybb_te[m]):.3f}", flush=True)

top class-1: ['a', 'nigger', 'the', 'you', 'that', 'not', 'are', 'kike', 'they', 'retarded']


top class-0: ['a', 'have', 'that', 'retarded', 'not', 'i', 'you', 'bitch', 'they', 'ghetto']


stopword mass share: 0.233


fidelity by target:


  None: n=685 fidelity=0.733


...

  Other: n=89 fidelity=0.730


  Refugee: n=78 fidelity=0.731


  Caucasian: n=50 fidelity=0.640


  Arab: n=47 fidelity=0.787


  Asian: n=31 fidelity=0.548


## Verdict

On this slice the surrogate tracks the box (fidelity ~0.76), ranks
rationale words well above chance (weighted AUROC ~0.65 vs ~0.49 random —
the canonical 300-epoch run reaches 0.71, about exact-SHAP parity), and
the faithfulness curves beat random when insertion is signed. It is a faithful
lexical distiller: good for auditing *which words* the classifier leans on
per community, not for syntax — and human agreement is reported as
plausibility, never as proof of correctness.